In [0]:
%pip install lightgbm
%pip install optuna
%restart_python

Looking in indexes: [REDACTED]
Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.
Looking in indexes: [REDACTED]
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/3.4 MB 78.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 611.9/611.9 kB 25.9 MB/s eta 0:00:00
Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


In [0]:
import numpy as np
import pandas as pd
from itertools import product
import mlflow
from lightgbm import LGBMRegressor
import optuna
from optuna.samplers import TPESampler
from mlflow.tracking import MlflowClient
import mlflow.lightgbm


## Baseline

In [0]:
df = spark.table("fmcg.gold.fact_sales_features").toPandas() #table is small enough to fit in the driver's memory no need for spark ML
df["date"] = pd.to_datetime(df["date"])
df = df.sort_values(["product_code", "channel", "date"]).reset_index(drop=True)
#No random splitting in time series !!

test_start = df["date"].max() - pd.DateOffset(months=2)
validation_start = test_start - pd.DateOffset(months=3)
# TRAIN: 2024-01 → 2025-06
# VALIDATION: 2025-07 → 2025-09
# TEST: 2025-10 → 2025-12
train = df[df["date"] < validation_start].copy()

validation = df[
    (df["date"] >= validation_start) & (df["date"] < test_start)
].copy()

test = df[df["date"] >= test_start].copy()

print("TRAIN:", train["date"].min(), "→", train["date"].max())
print("VALID:", validation["date"].min(), "→", validation["date"].max())
print("TEST :", test["date"].min(), "→", test["date"].max())

print(train.shape, validation.shape, test.shape)

TRAIN: 2024-04-01 00:00:00 → 2025-06-01 00:00:00
VALID: 2025-07-01 00:00:00 → 2025-09-01 00:00:00
TEST : 2025-10-01 00:00:00 → 2025-12-01 00:00:00
(8420, 18) (1942, 18) (1953, 18)


##### Baseline predictions

In [0]:
validation = validation.copy()
test = test.copy()

validation["baseline_pred"] = validation["lag_1"]
test["baseline_pred"] = test["lag_1"]

def mape(y_true, y_pred):
    y_true = np.array(y_true)
    y_pred = np.array(y_pred)

    mask = y_true != 0

    return (
        np.mean(
            np.abs(
                (y_true[mask] - y_pred[mask])
                / y_true[mask]
            )
        ) * 100
    )

validation_baseline_mape = mape(
    validation["sold_quantity"],
    validation["baseline_pred"]
)

test_baseline_mape = mape(
    test["sold_quantity"],
    test["baseline_pred"]
)

print(
    f"Validation Baseline MAPE: "
    f"{validation_baseline_mape:.2f}%" 
)
print(
    f"Test Baseline MAPE: "
    f"{test_baseline_mape:.2f}%"
)

Validation Baseline MAPE: 53.28%
Test Baseline MAPE: 43.65%


## Training

#### Defining feature sets 


In [0]:
# Base numerical/time features
base_features = [
    "lag_1",
    "lag_2",
    "lag_3",
    "month",
    "quarter",
    "price_inr"
]

# categorical features
base_categorical = [
    "product_code",
    "channel"
]

# Experiment 1: add more business dimensions
business_categorical = [
    "product_code",
    "channel",
    "market",
    "platform",
    "category"
]

# Experiment 2: add rolling mean
rolling_features = base_features + [
    "rolling_mean_3"
]

# Experiment 3: add yearly lag
lag12_features = base_features + [
    "lag_12"
]

# Experiment 4: add both
rolling_lag12_features = base_features + [
    "rolling_mean_3",
    "lag_12"
]

In [0]:
feature_experiments = {
    "base": {
        "features": base_features,
        "categoricals": base_categorical
    },

    "business_dimensions": {
        "features": base_features,
        "categoricals": business_categorical
    },

    "rolling_mean": {
        "features": rolling_features,
        "categoricals": base_categorical
    },

    "lag_12": {
        "features": lag12_features,
        "categoricals": base_categorical
    },

    "rolling_plus_lag12": {
        "features": rolling_lag12_features,
        "categoricals": base_categorical
    }
}

In [0]:
for exp in feature_experiments.values():

    for c in exp["categoricals"]:

        train[c] = train[c].astype("category")

        validation[c] = (
            validation[c]
            .astype("category")
            .cat.set_categories(train[c].cat.categories)
        )

        test[c] = (
            test[c]
            .astype("category")
            .cat.set_categories(train[c].cat.categories)
        )

#### Defining the hyperparameter grid

In [0]:
param_grid = {
    "n_estimators": [100, 200, 400],
    "learning_rate": [0.03, 0.05, 0.1],
    "max_depth": [4, 6, 8],
    "num_leaves": [15, 31, 50]
}
#3 × 3 × 3 × 3 = 81 combinations

In [0]:

param_combinations = [
    dict(zip(param_grid.keys(), values))
    for values in product(*param_grid.values())
]

print(
    f"Number of hyperparameter combinations: "
    f"{len(param_combinations)}"
)

Number of hyperparameter combinations: 81


In [0]:
mlflow.set_experiment(
    "/Users/fatine.achtouk2004@gmail.com/forcasting_model/optuna_search"
)

def objective(trial):
    feature_set_name = trial.suggest_categorical(
        "feature_set", list(feature_experiments.keys())
    )
    feature_cols = feature_experiments[feature_set_name]["features"]
    categorical_cols = feature_experiments[feature_set_name]["categoricals"]

    params = {
        "n_estimators": trial.suggest_int("n_estimators", 100, 500, step=50),
        "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.2, log=True),
        "max_depth": trial.suggest_int("max_depth", 3, 10),
        "num_leaves": trial.suggest_int("num_leaves", 15, 80),
    }

    X_train = train[feature_cols + categorical_cols]
    y_train = train["sold_quantity"]
    X_val = validation[feature_cols + categorical_cols]
    y_val = validation["sold_quantity"]

    with mlflow.start_run(nested=True):
        mlflow.log_param("feature_set", feature_set_name)
        mlflow.log_params(params)

        model = LGBMRegressor(**params, random_state=42, verbosity=-1)
        model.fit(X_train, y_train, categorical_feature=categorical_cols)

        preds = np.clip(model.predict(X_val), 0, None)
        val_mape = mape(y_val, preds)

        mlflow.log_metric("mape", val_mape)
        trial.set_user_attr("run_id", mlflow.active_run().info.run_id)

    return val_mape

study = optuna.create_study(
    direction="minimize",
    sampler=TPESampler(seed=42),
    study_name="lgbm_sales_forecast"
)

with mlflow.start_run(run_name="optuna_study_parent"):
    study.optimize(objective, n_trials=100, show_progress_bar=True)

print("Best MAPE:", study.best_value)
print("Best params:", study.best_params)

2026/08/21 17:05:29 INFO mlflow.tracking.fluent: Experiment with name '/Users/fatine.achtouk2004@gmail.com/forcasting_model/optuna_search' does not exist. Creating a new experiment.
[I 2026-08-21 17:05:29,893] A new study created in memory with name: lgbm_sales_forecast


  0%|          | 0/100 [00:00<?, ?it/s]

[I 2026-08-21 17:05:31,668] Trial 0 finished with value: 225.35776325413406 and parameters: {'feature_set': 'business_dimensions', 'n_estimators': 150, 'learning_rate': 0.011900590783184251, 'max_depth': 9, 'num_leaves': 54}. Best is trial 0 with value: 225.35776325413406.
[I 2026-08-21 17:05:32,817] Trial 1 finished with value: 135.32637555402175 and parameters: {'feature_set': 'rolling_mean', 'n_estimators': 150, 'learning_rate': 0.017322667470546258, 'max_depth': 5, 'num_leaves': 49}. Best is trial 1 with value: 135.32637555402175.
[I 2026-08-21 17:05:33,933] Trial 2 finished with value: 55.14095760524226 and parameters: {'feature_set': 'rolling_mean', 'n_estimators': 250, 'learning_rate': 0.03920673972242137, 'max_depth': 9, 'num_leaves': 28}. Best is trial 2 with value: 55.14095760524226.
[I 2026-08-21 17:05:34,765] Trial 3 finished with value: 53.65398767684154 and parameters: {'feature_set': 'lag_12', 'n_estimators': 100, 'learning_rate': 0.17160445029754812, 'max_depth': 10, 'n

### Selecting and registering the best model


In [0]:
best_params = study.best_params
best_feature_set = best_params.pop("feature_set")
feature_cols = feature_experiments[best_feature_set]["features"]
categorical_cols = feature_experiments[best_feature_set]["categoricals"]

X_train = train[feature_cols + categorical_cols]
y_train = train["sold_quantity"]
X_test = test[feature_cols + categorical_cols]
y_test = test["sold_quantity"]

final_model = LGBMRegressor(**best_params, random_state=42, verbosity=-1)
final_model.fit(X_train, y_train, categorical_feature=categorical_cols)

test_preds = np.clip(final_model.predict(X_test), 0, None)
test_mape = mape(y_test, test_preds)
print(f"TEST MAPE: {test_mape:.2f}%")

mlflow.set_registry_uri("databricks-uc")

with mlflow.start_run(run_name="final_champion_model") as run:
    mlflow.log_params({**best_params, "feature_set": best_feature_set})
    mlflow.log_metric("test_mape", test_mape)

    signature = mlflow.models.infer_signature(X_train, final_model.predict(X_train))

    model_info = mlflow.lightgbm.log_model(
        final_model,
        artifact_path="model",
        registered_model_name="fmcg.gold.sales_forecast_model",
        signature=signature
    )

client = MlflowClient()
client.set_registered_model_alias(
    "fmcg.gold.sales_forecast_model", "champion_model", model_info.registered_model_version
)

TEST MAPE: 32.51%


/databricks/python/lib/python3.12/site-packages/mlflow/types/utils.py:452: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/docs/latest/models.html#handling-integers-with-missing-values>`_ for more details.
  warnings.warn(
2026/08/21 17:16:52 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
🔗 View Logged Model at: https://dbc-da16ea83-8a78.cloud.databricks.com/ml/experiments/2386819912765509/models/m-7ed9d212c2fe4e368f992

Uploading artifacts:   0%|          | 0/9 [00:00<?, ?it/s]

🔗 Created version '2' of model 'fmcg.gold.sales_forecast_model': https://dbc-da16ea83-8a78.cloud.databricks.com/explore/data/models/fmcg/gold/sales_forecast_model/version/2?o=7474647516061225


In [0]:
results_df = pd.DataFrame(results)

results_df = (
    results_df
    .sort_values("mape")
    .reset_index(drop=True)
)

results_df.insert(
    0,
    "rank",
    range(1, len(results_df) + 1)
)

display(results_df)

rank,run_name,feature_set,n_estimators,learning_rate,max_depth,num_leaves,mape,mae,baseline_mape,improvement_vs_baseline,relative_improvement_pct,run_id
1,lag_12_lr0.03_depth4_leaves31_trees400,lag_12,400,0.03,4,31,44.66284004280918,699.8740267623067,53.27953547306228,8.616695430253102,16.172617410693523,b849148f911146559bf115662875b92e
2,lag_12_lr0.03_depth4_leaves50_trees400,lag_12,400,0.03,4,50,44.66284004280918,699.8740267623067,53.27953547306228,8.616695430253102,16.172617410693523,19f61c3f73dd4bb98af04d281dc2451d
3,base_lr0.1_depth8_leaves15_trees400,base,400,0.1,8,15,44.71194567941898,693.8861339854813,53.27953547306228,8.5675897936433,16.080451373257574,bd0aaf6a32994055927cba526c273b51
4,lag_12_lr0.05_depth4_leaves15_trees200,lag_12,200,0.05,4,15,44.876468034471515,695.9714626131614,53.27953547306228,8.403067438590767,15.771660477106247,774b967b34eb4a2f88c2d98a20c2a613
5,lag_12_lr0.1_depth8_leaves15_trees400,lag_12,400,0.1,8,15,45.00238582039996,703.1407654362775,53.27953547306228,8.277149652662324,15.535326235806215,65de441b6dec4d259f0227bb88c0627d
6,base_lr0.05_depth4_leaves15_trees200,base,200,0.05,4,15,45.237485712652656,689.6348655955486,53.27953547306228,8.042049760409625,15.094068837134705,b7b3585702074e8fb487f91f7df80fdc
7,rolling_plus_lag12_lr0.05_depth4_leaves31_trees400,rolling_plus_lag12,400,0.05,4,31,45.27566087332754,702.134459530261,53.27953547306228,8.003874599734743,15.022418136099253,3fbde1224afb41c2a901bffabb675e27
8,rolling_plus_lag12_lr0.05_depth4_leaves50_trees400,rolling_plus_lag12,400,0.05,4,50,45.27566087332754,702.134459530261,53.27953547306228,8.003874599734743,15.022418136099253,2deb31808bff4da4a63b40e42f61b923
9,rolling_mean_lr0.1_depth8_leaves15_trees400,rolling_mean,400,0.1,8,15,45.31846812426172,685.884560357515,53.27953547306228,7.961067348800562,14.94207349616555,aee092027a2f46bd9353b77009252681
10,base_lr0.03_depth4_leaves31_trees400,base,400,0.03,4,31,45.34385441081333,694.3283659869518,53.27953547306228,7.935681062248953,14.894426146528946,1e1f80412cf346b68eb5ce07eae0a9d8


###### Top 10

In [0]:
display(
    results_df.head(10)
) # lag_12 (yearly seasonality) is the best-performing feature addition

rank,run_name,feature_set,n_estimators,learning_rate,max_depth,num_leaves,mape,mae,baseline_mape,improvement_vs_baseline,relative_improvement_pct,run_id
1,lag_12_lr0.03_depth4_leaves31_trees400,lag_12,400,0.03,4,31,44.66284004280918,699.8740267623067,53.27953547306228,8.616695430253102,16.172617410693523,b849148f911146559bf115662875b92e
2,lag_12_lr0.03_depth4_leaves50_trees400,lag_12,400,0.03,4,50,44.66284004280918,699.8740267623067,53.27953547306228,8.616695430253102,16.172617410693523,19f61c3f73dd4bb98af04d281dc2451d
3,base_lr0.1_depth8_leaves15_trees400,base,400,0.1,8,15,44.71194567941898,693.8861339854813,53.27953547306228,8.5675897936433,16.080451373257574,bd0aaf6a32994055927cba526c273b51
4,lag_12_lr0.05_depth4_leaves15_trees200,lag_12,200,0.05,4,15,44.876468034471515,695.9714626131614,53.27953547306228,8.403067438590767,15.771660477106247,774b967b34eb4a2f88c2d98a20c2a613
5,lag_12_lr0.1_depth8_leaves15_trees400,lag_12,400,0.1,8,15,45.00238582039996,703.1407654362775,53.27953547306228,8.277149652662324,15.535326235806215,65de441b6dec4d259f0227bb88c0627d
6,base_lr0.05_depth4_leaves15_trees200,base,200,0.05,4,15,45.237485712652656,689.6348655955486,53.27953547306228,8.042049760409625,15.094068837134705,b7b3585702074e8fb487f91f7df80fdc
7,rolling_plus_lag12_lr0.05_depth4_leaves31_trees400,rolling_plus_lag12,400,0.05,4,31,45.27566087332754,702.134459530261,53.27953547306228,8.003874599734743,15.022418136099253,3fbde1224afb41c2a901bffabb675e27
8,rolling_plus_lag12_lr0.05_depth4_leaves50_trees400,rolling_plus_lag12,400,0.05,4,50,45.27566087332754,702.134459530261,53.27953547306228,8.003874599734743,15.022418136099253,2deb31808bff4da4a63b40e42f61b923
9,rolling_mean_lr0.1_depth8_leaves15_trees400,rolling_mean,400,0.1,8,15,45.31846812426172,685.884560357515,53.27953547306228,7.961067348800562,14.94207349616555,aee092027a2f46bd9353b77009252681
10,base_lr0.03_depth4_leaves31_trees400,base,400,0.03,4,31,45.34385441081333,694.3283659869518,53.27953547306228,7.935681062248953,14.894426146528946,1e1f80412cf346b68eb5ce07eae0a9d8
